In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import os
!pip install openpyxl
from openpyxl import load_workbook

yesterday = (datetime.today() - timedelta(days=1)).strftime('%d%m%y')

In [ ]:
def load_data(df_title_manual=None):
    today = datetime.today().strftime("%d%m%y")
    yesterday = (datetime.today() - timedelta(days=1)).strftime("%d%m%y")

    try:
        # Tentukan nama file master branch
        if df_title_manual:
            df_master = df_title_manual
        else:
            df_master = f"Data {yesterday}_ZBRI_SRC_RO_SMNT.XLSX"

        # Tentukan nama file sumber lainnya
        src = f"Data {yesterday}_ZBRI_SRC_RO_SMNT.XLSX"
        Dmedium = "Daily Medium Corporate NDS Tagging 080825.xlsx"

        # Validasi keberadaan file
        if not os.path.exists(src):
            raise FileNotFoundError(f"❌ File tidak ditemukan: {src}")
        if not os.path.exists(Dmedium):
            raise FileNotFoundError(f"❌ File tidak ditemukan: {Dmedium}")

        # Load file excel
        src = pd.read_excel(src)
        Dmedium = pd.read_excel(Dmedium, header=2)

        # Bersihkan kolom tidak perlu
        if "Unnamed: 0" in Dmedium.columns:
            Dmedium = Dmedium.drop(columns=["Unnamed: 0"])

        return src, Dmedium

    except Exception as e:
        print(f"⚠️ Gagal load data: {e}")
        return None, None

def merge_data(src, Dmedium):
  try:
    R_Src_RO = pd.merge(
        src,
        Dmedium[['RO Code', 'RO DESCRIPTION', 'SEGMENT']],
        left_on='Revenue Owner',
        right_on='RO Code',
        how="left"
    )
    return R_Src_RO

  except Exception as e:
    print(f"⚠️ Gagal merge data: {e}")
    return None

In [ ]:
def filter_data(R_Src_RO):
  if R_Src_RO is None:
    return None

  try:
    R_Src_RO = R_Src_RO.rename(columns={
        "RO DESCRIPTION": "RO Description",
        "SEGMENT": "Segment Description"
    })

    columns_order = [
          'Date','External Number - Source','External Number','Revenue Owner',
          'RO Description','Business Segment','Segment Description',
          'Rev Own & Segment Flag','Source System Basic Data','Changed by',
          'Changed On','Time of change'
    ]

    existing_cols = [col for col in columns_order if col in R_Src_RO.columns]
    return R_Src_RO[existing_cols]

  except Exception as e:
    print(f"⚠️ Gagal filter data: {e}")
    return None

In [ ]:
def save_result(R_Src_RO):
    if R_Src_RO is None:
        print("❌ Tidak ada data untuk disimpan.")
        return
    filename = f"{yesterday}_ZBRI_SRC_RO_SMNT.xlsx"
    try:
        R_Src_RO.to_excel(filename, index=False, engine="openpyxl")

        wb = load_workbook(filename)
        ws = wb.active

        for col in ws.columns:
            max_length = 0
            col_letter = col[0].column_letter

            for cell in col:
                try:
                    if cell.value:
                        length = len(str(cell.value))
                        if length > max_length:
                            max_length = length
                except Exception:
                    pass

            adjusted_width = max_length + 2
            ws.column_dimensions[col_letter].width = adjusted_width

        wb.save(filename)
        print(f"📂 File berhasil disimpan: {filename}")
    except Exception as e:
        print(f"⚠️ Gagal simpan file: {e}")

In [ ]:
def main():
    src, Dmedium = load_data()
    # src, Dmedium = load_data('241125')
    df_result = merge_data(src, Dmedium)
    df_result = filter_data(df_result)
    save_result(df_result)

main()

📂 File berhasil disimpan: 210526_ZBRI_SRC_RO_SMNT.xlsx
